In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpretation-Conditioned Autoregressive Artificial Graph Generation
Build base graphs under a fixed, immutable interpretation graph context.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
from abstractgraph.graphs import AbstractGraph, graph_to_abstract_graph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.labels import graph_hash_label_function_factory
from abstractgraph.operators import *
from abstractgraph import ArtificialGraphDatasetConstructor

def draw(graph, df, nbits=10):
    display_decomposition_graph(df)
    ag = graph_to_abstract_graph(graph, decomposition_function=df, nbits=nbits)
    display(ag, size=(12,6))
    display_mappings(ag, n_elements_per_row=10)

In [ ]:
def estimated_predictive_performance(train_graphs, train_targets, test_graphs, test_targets):
    from nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
    train_X = vectorizer.transform(train_graphs)
    test_X = vectorizer.transform(test_graphs)
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    clf.fit(train_X, train_targets)
    from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
    y_prob = clf.predict_proba(test_X)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(test_targets, y_pred)
    print(f'Test accuracy: {acc:.3f} | ROC-AUC: {roc_auc_score(test_targets, y_prob):.3f} | AvgPrec: {average_precision_score(test_targets, y_prob):.3f}')

In [ ]:
def estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets):
    from nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    from sklearn.ensemble import ExtraTreesClassifier
    classifier = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)

    from abstractgraph_generative.generative_performance import compute_expected_gain_weighted_equivalent_data_size
    results = compute_expected_gain_weighted_equivalent_data_size(
        generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets,
        vectorizer=vectorizer, classifier=classifier,
        fracional_size=(1,2,3,4,5,7,10,15),
        n_repeats=30,
    )
    print(f'Expected gain weighted equivalent data size:{results["expected_gain_weighted_equivalent_data_size"]:.3f}')

    from abstractgraph_generative.generative_performance import plot_expected_gain_weighted_equivalent_data_size
    _ = plot_expected_gain_weighted_equivalent_data_size(results)

---

In [ ]:
def offset_neg_graphs(graphs, targets, offset=10):
    out_graphs = []
    for graph, target in zip(graphs, targets):
        if target == 0:
            for u in graph.nodes():
                graph.nodes[u]['label'] += offset
        out_graphs.append(graph.copy())
    return out_graphs, targets

def select_pos_neg(sampled_graphs, sampled_targets, n_lines=3, n_graphs_per_line=12):
    import random
    k = n_graphs_per_line * n_lines
    pos_candidates = [sampled_graph for sampled_graph, sampled_target in zip(sampled_graphs, sampled_targets) if sampled_target == 1]
    neg_candidates = [sampled_graph for sampled_graph, sampled_target in zip(sampled_graphs, sampled_targets) if sampled_target != 1]
    pos_graphs = random.sample(pos_candidates, k=min(k, len(pos_candidates)))
    neg_graphs = random.sample(neg_candidates, k=min(k, len(neg_candidates)))
    return pos_graphs, neg_graphs

dataset_size = 100
alphabet_size = 4
size = 10
graph_types = ['path', 'tree', 'cycle', 'degree', 'regular', 'dense']

graphs, targets = ArtificialGraphDatasetConstructor(
    graph_generator_target_type_pos='cycle',
    graph_generator_context_type_pos='cycle',
    graph_generator_target_type_neg='path',
    graph_generator_context_type_neg='path',
    target_size_pos=size,
    context_size_pos=size,
    n_link_edges_pos=1,
    alphabet_size_pos=alphabet_size,
    target_size_neg=size,
    context_size_neg=size,
    n_link_edges_neg=1,
    alphabet_size_neg=alphabet_size,
).sample(dataset_size // 2)

graphs, targets = offset_neg_graphs(graphs, targets, offset=alphabet_size+1)

print('#graphs:%d'%(len(graphs)))

n_graphs_per_line = 10
n_lines = 2
num_graphs = n_lines * n_graphs_per_line
pos_graphs, neg_graphs = select_pos_neg(graphs, targets, n_lines=2, n_graphs_per_line=n_graphs_per_line)

from abstractgraph.display import display_graphs
display_graph_size=2
_ = display_graphs(neg_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))
_ = display_graphs(pos_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))

---

In [ ]:
draw(graphs[-2], decomposition_function, nbits=10)
draw(graphs[2], decomposition_function, nbits=10)

In [ ]:
%%time
from abstractgraph_generative.conditional_batch import ConditionalAutoregressiveGraphsGenerator

dataset_generator = ConditionalAutoregressiveGraphsGenerator(
    generator=generator,
    graph_estimator=graph_estimator,
    verbose=True,
)

dataset_generator.fit(graphs, targets)
generated_graphs, generated_targets = dataset_generator.generate(
    n_samples=num_graphs*2,
    max_attempts_per_sample=24,
)

In [ ]:
pos_generated_graphs, neg_generated_graphs = select_pos_neg(generated_graphs, generated_targets, n_lines=n_lines, n_graphs_per_line=n_graphs_per_line)
print(f'#graphs={len(generated_graphs)} #pos={len(pos_generated_graphs)}, #neg={len(neg_generated_graphs)}  ')
from abstractgraph.display import display_graphs
_ = display_graphs(neg_generated_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))
_ = display_graphs(pos_generated_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))

In [ ]:
pos_graphs, neg_graphs = select_pos_neg(graphs, targets, n_lines=n_lines, n_graphs_per_line=n_graphs_per_line)
_ = display_graphs(neg_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))
_ = display_graphs(pos_graphs, n_graphs_per_line=n_graphs_per_line, size=(display_graph_size,display_graph_size))

---